# HW2: Who should the bank call?

**Name:** SINA BAGHERZADEH KHIAVI &nbsp;&nbsp; **Student ID:** N26120261

See `README.md` for the scenario, requirements and grading. This starter loads the data, sets up MLflow and writes the submission files; the rest is yours. Put your final model in `hw2_submission.py`.

In [1]:
import os
os.environ["OMP_NUM_THREADS"] = "2"   # keep OpenMP threads modest on shared machines
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"  # quieter MLflow logs

# Silence noisy library warnings / INFO logs for the whole notebook
import logging
import warnings
warnings.filterwarnings("ignore")
for _name in [
    "mlflow",
    "mlflow.tracking",
    "mlflow.store",
    "mlflow.utils",
    "urllib3",
    "alembic",
    "sqlalchemy",
]:
    logging.getLogger(_name).setLevel(logging.ERROR)
logging.getLogger().setLevel(logging.ERROR)

import json

import mlflow
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_openml

import hw2_submission as sub

SEED = 0
EXPERIMENT = "hw2-call-list"
# Absolute path avoids "readonly database" when the notebook cwd is not the project folder
_MLFLOW_DB = os.path.abspath("mlflow.db")
mlflow.set_tracking_uri(f"sqlite:///{_MLFLOW_DB}")
# browse later with: mlflow ui --backend-store-uri sqlite:///<absolute-path-to>/mlflow.db
mlflow.set_experiment(EXPERIMENT)
print("MLflow DB:", _MLFLOW_DB)

/Users/hemekci/Desktop/Hacettepe/AIE683-Machine-Learning/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<Experiment: artifact_location='/Users/hemekci/Desktop/Hacettepe/AIE683-Machine-Learning/homeworks/hw2-call-list/mlruns/1', creation_time=1790703362663, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1790703362663, lifecycle_stage='active', name='hw2-call-list', tags={}, trace_location=None, workspace='default'>

In [2]:
bank = fetch_openml(data_id=1461, as_frame=True)
NAMES = ["age", "job", "marital", "education", "default", "balance", "housing", "loan", "contact",
         "day", "month", "duration", "campaign", "pdays", "previous", "poutcome"]
X = bank.data.copy()
X.columns = NAMES                                  # OpenML stores them as V1..V16
y = (bank.target == "2").astype(int).to_numpy()    # 1 = subscribed to a term deposit
for c in X.select_dtypes(exclude="number").columns:
    X[c] = X[c].astype("category")                 # exactly what the grader passes to your pipeline
print(X.shape, f"positive rate {y.mean():.3f}  (rows are in chronological order)")
X.head()

(45211, 16) positive rate 0.117  (rows are in chronological order)


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown


### Quick look at the data

Portuguese bank **telemarketing** data (OpenML bank-marketing). Each row is one past phone contact. Target `y`: **1 = subscribed** to a term deposit, **0 = did not** (~11.7% positive).

| Column | Meaning |
|---|---|
| `age` | Client age in years |
| `job` | Job type (admin, blue-collar, technician, …) |
| `marital` | Marital status (married / single / divorced) |
| `education` | Education level (primary / secondary / tertiary / unknown) |
| `default` | Has credit in default? (`yes` / `no`) |
| `balance` | Average yearly bank balance (euros) |
| `housing` | Has housing loan? (`yes` / `no`) |
| `loan` | Has personal loan? (`yes` / `no`) |
| `contact` | Contact channel (`cellular` / `telephone` / `unknown`) |
| `day` | Day of month of last contact |
| `month` | Month of last contact |
| `duration` | Last call length in **seconds** — known only *after* the call (classic **leak** for ranking who to call) |
| `campaign` | Number of contacts during this campaign (includes this one) |
| `pdays` | Days since previous campaign contact (`-1` = never contacted before) |
| `previous` | Number of contacts before this campaign |
| `poutcome` | Outcome of the previous campaign (`success` / `failure` / `other` / `unknown`) |

In [ ]:
import matplotlib.pyplot as plt

# Attach label so we can compare subscribers vs non-subscribers
df = X.copy()
df["subscribed"] = y

print("dtypes:")
print(df.dtypes)
print("\nmissing values:")
print(df.isna().sum())
print("\nnumeric summary:")
display(df.describe().T)
print("\ncategorical value counts (top levels):")
for c in df.select_dtypes(include="category").columns:
    print(f"\n{c}:")
    print(df[c].value_counts().head(8).to_string())

# --- plots ---
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.ravel()

# 1) class balance
axes[0].bar(["no (0)", "yes (1)"], [(y == 0).sum(), (y == 1).sum()], color=["#7f8c8d", "#2ecc71"])
axes[0].set_title("Target: subscribed?")
axes[0].set_ylabel("count")

# 2) age by subscription
for label, color in [(0, "#7f8c8d"), (1, "#2ecc71")]:
    axes[1].hist(df.loc[df["subscribed"] == label, "age"], bins=25, alpha=0.55,
                 label=f"y={label}", color=color, density=True)
axes[1].set_title("Age by subscription")
axes[1].set_xlabel("age")
axes[1].legend()

# 3) balance (clip extreme outliers for readability)
bal = df["balance"].clip(-2000, 10000)
for label, color in [(0, "#7f8c8d"), (1, "#2ecc71")]:
    axes[2].hist(bal[df["subscribed"] == label], bins=30, alpha=0.55,
                 label=f"y={label}", color=color, density=True)
axes[2].set_title("Balance (clipped) by subscription")
axes[2].set_xlabel("balance (€)")
axes[2].legend()

# 4) duration — strong signal, but a leak for call ranking
for label, color in [(0, "#7f8c8d"), (1, "#2ecc71")]:
    axes[3].hist(df.loc[df["subscribed"] == label, "duration"].clip(0, 1500),
                 bins=30, alpha=0.55, label=f"y={label}", color=color, density=True)
axes[3].set_title("Call duration (clipped) — leak candidate")
axes[3].set_xlabel("seconds")
axes[3].legend()

# 5) job mix
job_rates = df.groupby("job", observed=True)["subscribed"].mean().sort_values()
axes[4].barh(job_rates.index.astype(str), job_rates.values, color="#3498db")
axes[4].set_title("Subscription rate by job")
axes[4].set_xlabel("P(y=1)")

# 6) month mix
month_order = ["jan","feb","mar","apr","may","jun","jul","aug","sep","oct","nov","dec"]
month_rates = df.groupby("month", observed=True)["subscribed"].mean().reindex(month_order)
axes[5].bar(month_rates.index.astype(str), month_rates.values, color="#9b59b6")
axes[5].set_title("Subscription rate by month")
axes[5].tick_params(axis="x", rotation=45)
axes[5].set_ylabel("P(y=1)")

plt.tight_layout()
plt.show()

# correlation among numeric features (+ target)
num = df.select_dtypes(include="number")
corr = num.corr()
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=45, ha="right")
ax.set_yticklabels(corr.columns)
ax.set_title("Numeric feature correlations")
fig.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()

## 1. Data and protocol

**Goal:** decide the rules *before* training models.

### Plan

1. **Find the leak with evidence** (next cell): which column is unavailable at call-list time *and* inflates AUC if included?
2. **Freeze features + chronological split** once the leak is identified.
3. **Tuning plan (used in §2):** equal budget of random hyperparameter configs per model family; select by validation **ROC AUC**; fixed `SEED`; log every search to MLflow.

### Split (after leak is fixed)

Rows are chronological, so we split **by time**, not at random:

- first **60%** → train
- next **20%** → validation (hyperparameter search)
- last **20%** → held-out test (final comparison only)

### 1a. Find the leak (evidence, not just intuition)

We look for a column that is (A) **not known when building the call list**, and (B) **unusually predictive** if we sneak it into a model.

Method:
1. Score every column alone with a tiny logistic regression → single-feature validation AUC.
2. The top column is the main leak *candidate*.
3. Prove the damage: same model on **all features including that column** vs **all features excluding it**.

In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Same time order for this diagnostic: first 60% train, next 20% val
# (test is left untouched so final evaluation stays honest)
n_all = len(X)
cut_tr = int(0.60 * n_all)
cut_va = int(0.80 * n_all)
X_tr_all, y_tr = X.iloc[:cut_tr], y[:cut_tr]
X_va_all, y_va = X.iloc[cut_tr:cut_va], y[cut_tr:cut_va]


def make_lr_pipeline(frame_like):
    """Small LR pipeline that one-hots categories and scales numbers."""
    cat_cols = frame_like.select_dtypes(include="category").columns.tolist()
    num_cols = frame_like.select_dtypes(include="number").columns.tolist()
    transformers = []
    if cat_cols:
        transformers.append(("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols))
    if num_cols:
        transformers.append(("num", StandardScaler(), num_cols))
    pre = ColumnTransformer(transformers=transformers)
    return Pipeline([
        ("pre", pre),
        ("clf", LogisticRegression(max_iter=2000, random_state=SEED)),
    ])


def val_auc(cols):
    pipe = make_lr_pipeline(X_tr_all[cols])
    pipe.fit(X_tr_all[cols], y_tr)
    proba = pipe.predict_proba(X_va_all[cols])[:, 1]
    return float(roc_auc_score(y_va, proba))


# --- A) decision-time availability (domain check) ---
# Only "after the call" fails the pre-call protocol.
availability = {
    "age": "before call",
    "job": "before call",
    "marital": "before call",
    "education": "before call",
    "default": "before call",
    "balance": "before call",
    "housing": "before call",
    "loan": "before call",
    "contact": "before / at dial time",
    "day": "before / at dial time",
    "month": "before / at dial time",
    "duration": "AFTER the call ends",
    "campaign": "during campaign (known before this dial for ranking)",
    "pdays": "before call (past campaigns)",
    "previous": "before call (past campaigns)",
    "poutcome": "before call (past campaigns)",
}

# --- B) single-feature predictive power ---
single_auc = {}
for col in X.columns:
    single_auc[col] = val_auc([col])

single_auc_sorted = sorted(single_auc.items(), key=lambda kv: kv[1], reverse=True)
print("Single-feature validation AUC (higher = more predictive alone):")
for col, auc in single_auc_sorted:
    flag = "  <-- AFTER-CALL" if availability[col].startswith("AFTER") else ""
    print(f"  {col:12s}  AUC={auc:.3f}  [{availability[col]}]{flag}")

leak_candidate = single_auc_sorted[0][0]
print(f"\nTop predictive column: {leak_candidate} "
      f"(AUC={single_auc[leak_candidate]:.3f})")

# Require both: strongest signal AND unavailable at decision time
assert availability[leak_candidate].startswith("AFTER"), (
    f"Top column {leak_candidate!r} is predictive but still available "
    f"pre-call — revisit the leak diagnosis."
)

# --- C) quantify the inflation: all features WITH vs WITHOUT the candidate ---
auc_with = val_auc(X.columns.tolist())
auc_without = val_auc([c for c in X.columns if c != leak_candidate])
gap = auc_with - auc_without

print("\nProof — same logistic regression, same val split:")
print(f"  all features INCLUDING {leak_candidate}: AUC = {auc_with:.3f}")
print(f"  all features EXCLUDING {leak_candidate}: AUC = {auc_without:.3f}")
print(f"  inflation (gap)                    : {gap:+.3f}")

LEAK_COLUMN = leak_candidate
print(f"\n=> Identified leak column: {LEAK_COLUMN}")

/opt/homebrew/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/opt/homebrew/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: overflow encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/opt/homebrew/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: invalid value encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/opt/homebrew/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/opt/homebrew/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/opt/homebrew/lib/python

Single-feature validation AUC (higher = more predictive alone):
  duration      AUC=0.768  [AFTER the call ends]  <-- AFTER-CALL
  housing       AUC=0.625  [before call]
  job           AUC=0.582  [before call]
  balance       AUC=0.577  [before call]
  pdays         AUC=0.574  [before call (past campaigns)]
  education     AUC=0.571  [before call]
  previous      AUC=0.565  [before call (past campaigns)]
  poutcome      AUC=0.564  [before call (past campaigns)]
  month         AUC=0.556  [before / at dial time]
  loan          AUC=0.536  [before call]
  campaign      AUC=0.528  [during campaign (known before this dial for ranking)]
  marital       AUC=0.526  [before call]
  age           AUC=0.508  [before call]
  default       AUC=0.499  [before call]
  contact       AUC=0.489  [before / at dial time]
  day           AUC=0.442  [before / at dial time]

Top predictive column: duration (AUC=0.768)

Proof — same logistic regression, same val split:
  all features INCLUDING duration: AUC

/opt/homebrew/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/opt/homebrew/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/opt/homebrew/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


In [3]:
# --- 1) features = everything except the empirically identified leak ---
# LEAK_COLUMN comes from the proof cell above (should be "duration")
assert "LEAK_COLUMN" in dir(), "Run the leak-proof cell first"
FEATURES = [c for c in X.columns if c != LEAK_COLUMN]
assert LEAK_COLUMN not in FEATURES

X_model = X[FEATURES].copy()
y_model = y.copy()

print(f"leak excluded: {LEAK_COLUMN}")
print(f"n_features: {len(FEATURES)}")
print("FEATURES:", FEATURES)

# --- 2) chronological 60 / 20 / 20 split (rows already time-ordered) ---
n = len(X_model)
i_train_end = int(0.60 * n)
i_val_end = int(0.80 * n)

X_train, y_train = X_model.iloc[:i_train_end], y_model[:i_train_end]
X_val, y_val = X_model.iloc[i_train_end:i_val_end], y_model[i_train_end:i_val_end]
X_test, y_test = X_model.iloc[i_val_end:], y_model[i_val_end:]

# keep indices handy for later MLflow / debugging
split_sizes = {
    "train": (len(X_train), float(y_train.mean())),
    "val": (len(X_val), float(y_val.mean())),
    "test": (len(X_test), float(y_test.mean())),
}
print("\nsplit sizes and positive rates (chronological 60/20/20):")
for name, (size, rate) in split_sizes.items():
    print(f"  {name:5s}: n={size:5d}  positive_rate={rate:.3f}")

# --- 3) protocol knobs for §2 ---
N_CONFIGS = 20          # same # of hyperparameter trials for EVERY model family
SCORING = "roc_auc"     # primary selection metric
print(f"\ntuning: {N_CONFIGS} configs/family, metric={SCORING}, seed={SEED}")

## 2. Candidates, baseline and equal-budget tuning

**Goal:** train several model *families* under the **same tuning budget**, pick the best config of each by validation ROC AUC, then compare them fairly.

### Families we compare
| Family | Role |
|---|---|
| `logistic_regression` | **Baseline** (simple linear model) |
| `random_forest` | Bagged trees |
| `hist_gradient_boosting` | sklearn gradient boosting |
| `lightgbm` | Another boosting library |
| `mlp` | Small neural net |

### Equal budget
Each family gets exactly `N_CONFIGS` random hyperparameter tries (set in §1). Winner of a family = highest **validation ROC AUC**. We also track **lift@top10%** (call-list metric).

### Pipelines
- Linear / MLP need **one-hot encoding + scaling**
- Tree models need **ordinal encoding** of categories (trees split on codes; scaling not required)

Every trial is logged to **MLflow** (`experiment = hw2-call-list`).

In [4]:
# =============================================================================
# §2 — Equal-budget model comparison
# =============================================================================
# What this cell does, in order:
#   1) Define helper metrics (AUC already known; lift@top10% for the call list)
#   2) Build preprocessing that matches each model family
#   3) Define each candidate model + which hyperparameters we will randomly try
#   4) For EVERY family, try exactly N_CONFIGS settings on (X_train, y_train),
#      score on (X_val, y_val), and log each try to MLflow
#   5) Keep the best pipeline per family and print a comparison table
# =============================================================================

from copy import deepcopy

import mlflow
from lightgbm import LGBMClassifier
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import ParameterSampler
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

# warnings / logging already silenced in the setup cell

# ---------------------------------------------------------------------------
# 1) Call-list metric: lift in the top 10%
# ---------------------------------------------------------------------------
# Sort clients by model score (high → low). Look at the top 10%.
# Lift = (subscription rate in that top slice) / (overall subscription rate).
# Lift = 1  → no better than calling at random
# Lift = 3  → top slice has 3× more "yes" clients than average
def lift_at_top_k(y_true, scores, k=0.10):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)
    n_top = max(1, int(k * len(y_true)))
    top_idx = np.argsort(scores)[::-1][:n_top]   # highest scores first
    overall_rate = y_true.mean()
    if overall_rate == 0:
        return float("nan")
    return float(y_true[top_idx].mean() / overall_rate)


# ---------------------------------------------------------------------------
# 2) Preprocessing builders (fit ONLY on training data inside the Pipeline)
# ---------------------------------------------------------------------------
# Column types from our feature matrix (duration already removed in §1)
CAT_COLS = X_train.select_dtypes(include="category").columns.tolist()
NUM_COLS = X_train.select_dtypes(include="number").columns.tolist()
print("categorical:", CAT_COLS)
print("numeric    :", NUM_COLS)


def make_linear_preprocess():
    """For logistic regression & MLP: categories → one-hot, numbers → scale."""
    return ColumnTransformer(
        transformers=[
            ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_COLS),
            ("num", StandardScaler(), NUM_COLS),
        ]
    )


def make_tree_preprocess():
    """For tree models: categories → integer codes. Numbers pass through."""
    # unknown_value=-1 handles any rare category that appears only in val/test
    return ColumnTransformer(
        transformers=[
            (
                "cat",
                OrdinalEncoder(
                    handle_unknown="use_encoded_value",
                    unknown_value=-1,
                ),
                CAT_COLS,
            ),
            ("num", "passthrough", NUM_COLS),
        ]
    )


def wrap(preprocess, clf):
    """Glue preprocess + classifier into one Pipeline object."""
    return Pipeline([("pre", preprocess), ("clf", clf)])


# ---------------------------------------------------------------------------
# 3) Model families: estimator factory + hyperparameter search space
# ---------------------------------------------------------------------------
# ParameterSampler will draw N_CONFIGS random combinations from each space.
# Keys use the "clf__" prefix because the model sits in the Pipeline step "clf".
FAMILIES = {
    # --- baseline ----------------------------------------------------------
    "logistic_regression": {
        "build": lambda: wrap(
            make_linear_preprocess(),
            LogisticRegression(max_iter=2000, random_state=SEED),
        ),
        "space": {
            "clf__C": np.logspace(-3, 2, 20),          # regularization strength
            "clf__penalty": ["l2"],
            "clf__solver": ["lbfgs"],
            "clf__class_weight": [None, "balanced"],  # help with rare "yes" class
        },
    },
    # --- bagged trees ------------------------------------------------------
    "random_forest": {
        "build": lambda: wrap(
            make_tree_preprocess(),
            RandomForestClassifier(random_state=SEED, n_jobs=2),
        ),
        "space": {
            "clf__n_estimators": [100, 200, 300],
            "clf__max_depth": [None, 8, 16, 24],
            "clf__min_samples_leaf": [1, 2, 5, 10],
            "clf__max_features": ["sqrt", "log2", 0.5],
            "clf__class_weight": [None, "balanced_subsample"],
        },
    },
    # --- sklearn gradient boosting -----------------------------------------
    "hist_gradient_boosting": {
        "build": lambda: wrap(
            make_tree_preprocess(),
            HistGradientBoostingClassifier(random_state=SEED),
        ),
        "space": {
            "clf__learning_rate": [0.03, 0.05, 0.1, 0.2],
            "clf__max_depth": [3, 5, 8, None],
            "clf__max_leaf_nodes": [15, 31, 63],
            "clf__min_samples_leaf": [10, 20, 50],
            "clf__l2_regularization": [0.0, 0.1, 1.0],
        },
    },
    # --- LightGBM boosting -------------------------------------------------
    "lightgbm": {
        "build": lambda: wrap(
            make_tree_preprocess(),
            LGBMClassifier(
                random_state=SEED,
                n_jobs=2,
                verbosity=-1,   # keep logs quiet
            ),
        ),
        "space": {
            "clf__n_estimators": [100, 200, 400],
            "clf__learning_rate": [0.03, 0.05, 0.1, 0.2],
            "clf__num_leaves": [15, 31, 63],
            "clf__min_child_samples": [10, 20, 50],
            "clf__subsample": [0.7, 0.9, 1.0],
            "clf__colsample_bytree": [0.7, 0.9, 1.0],
        },
    },
    # --- small neural net --------------------------------------------------
    "mlp": {
        "build": lambda: wrap(
            make_linear_preprocess(),
            MLPClassifier(
                random_state=SEED,
                max_iter=200,          # keep runtime under the 20-min budget
                early_stopping=True,
            ),
        ),
        "space": {
            "clf__hidden_layer_sizes": [(64,), (128,), (64, 32), (128, 64)],
            "clf__alpha": [1e-5, 1e-4, 1e-3, 1e-2],   # L2 penalty
            "clf__learning_rate_init": [1e-3, 3e-3, 1e-2],
        },
    },
}


# ---------------------------------------------------------------------------
# 4) Equal-budget random search + MLflow logging
# ---------------------------------------------------------------------------
def run_family_search(family_name, family_spec, n_configs=N_CONFIGS, seed=SEED):
    """Try n_configs random hyperparameter settings for one model family.

    Returns
    -------
    best : dict with keys
        pipeline, params, val_auc, val_lift, run_id
    """
    # Draw the SAME NUMBER of configs for every family → fair comparison
    sampler = list(
        ParameterSampler(family_spec["space"], n_iter=n_configs, random_state=seed)
    )

    best = {"val_auc": -np.inf}

    # One parent MLflow run per family; each config is a nested child run
    with mlflow.start_run(run_name=f"search_{family_name}") as parent:
        mlflow.set_tags({"family": family_name, "stage": "tuning"})
        mlflow.log_param("n_configs", n_configs)

        for i, params in enumerate(sampler):
            pipe = family_spec["build"]()
            pipe.set_params(**params)

            with mlflow.start_run(run_name=f"{family_name}_cfg{i:02d}", nested=True) as child:
                # --- train on train split only ---
                pipe.fit(X_train, y_train)

                # --- score on validation split (not the test set!) ---
                val_scores = pipe.predict_proba(X_val)[:, 1]
                val_auc = float(roc_auc_score(y_val, val_scores))
                val_lift = lift_at_top_k(y_val, val_scores, k=0.10)

                # log params + metrics so answers.json can point at run_ids later
                mlflow.log_params({k: str(v) for k, v in params.items()})
                mlflow.log_metrics({"val_auc": val_auc, "val_lift_top10": val_lift})
                mlflow.set_tags({"family": family_name, "config_index": i})

                # keep the best validation AUC for this family
                if val_auc > best["val_auc"]:
                    best = {
                        "family": family_name,
                        "pipeline": deepcopy(pipe),  # freeze fitted weights
                        "params": params,
                        "val_auc": val_auc,
                        "val_lift": val_lift,
                        "run_id": child.info.run_id,
                        "parent_run_id": parent.info.run_id,
                    }

            print(
                f"  [{family_name:22s} cfg {i:02d}/{n_configs - 1:02d}]  "
                f"val_auc={val_auc:.4f}  lift@10%={val_lift:.3f}"
            )

        # record the winner on the parent run too
        mlflow.log_metrics({
            "best_val_auc": best["val_auc"],
            "best_val_lift_top10": best["val_lift"],
        })
        mlflow.log_param("best_child_run_id", best["run_id"])

    return best


print(f"Equal budget: {N_CONFIGS} configs × {len(FAMILIES)} families "
      f"(seed={SEED})\n")

best_by_family = {}
for name, spec in FAMILIES.items():
    print(f"\n=== tuning {name} ===")
    best_by_family[name] = run_family_search(name, spec)

# ---------------------------------------------------------------------------
# 5) Summary table — who won the validation bake-off?
# ---------------------------------------------------------------------------
summary_rows = []
for name, info in best_by_family.items():
    summary_rows.append({
        "family": name,
        "val_auc": info["val_auc"],
        "val_lift_top10": info["val_lift"],
        "mlflow_run_id": info["run_id"],
    })

summary = (
    pd.DataFrame(summary_rows)
    .sort_values("val_auc", ascending=False)
    .reset_index(drop=True)
)
print("\n=== validation leaderboard (equal budget) ===")
display(summary)

# Best family on validation (used as the candidate for §3 test evaluation)
VAL_WINNER = summary.iloc[0]["family"]
print(f"\nValidation winner: {VAL_WINNER}  "
      f"(AUC={summary.iloc[0]['val_auc']:.4f}, "
      f"lift@10%={summary.iloc[0]['val_lift_top10']:.3f})")

# Keep a convenient handle for the baseline (needed in §3 for the CI)
BASELINE_NAME = "logistic_regression"
assert BASELINE_NAME in best_by_family

## 3. Final evaluation with uncertainty

**Goal:** check the §2 winner **once** on the held-out **test** set, and ask: is it really better than the logistic baseline, or was validation just lucky?

### Steps
1. Refit the validation winner and the logistic baseline on **train+val** (using their best hyperparameters from §2).
2. Score both on the untouched **test** set → test AUC and lift@top10%.
3. Build a **95% paired bootstrap CI** for `(winner AUC − baseline AUC)` on that same test set.

### How to read the CI
- If the interval is **entirely above 0** → evidence the winner beats the baseline.
- If it **includes 0** → the difference is not clearly real; prefer the simpler baseline.
- If it is **entirely below 0** → baseline is better.

### Bootstrap helper (provided by the starter)

`paired_bootstrap_ci(y_true, score_a, score_b)` resamples the **same** test rows many times and recomputes `metric(A) − metric(B)` on each resample. Keeping the pairing (same clients for both models) makes the comparison fairer than two independent intervals.


In [5]:
import numpy as np
from sklearn.metrics import roc_auc_score


def paired_bootstrap_ci(y_true, score_a, score_b, metric=roc_auc_score, n_boot=2000, seed=0, level=0.95):
    """Bootstrap interval for metric(A) - metric(B) on the same test set.

    Resamples test rows with replacement and recomputes both metrics on each
    resample, so the pairing between the two models is kept.
    Returns (observed difference, lower bound, upper bound).
    """
    y_true, score_a, score_b = map(np.asarray, (y_true, score_a, score_b))
    rng = np.random.default_rng(seed)
    n = len(y_true)
    diffs = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        if len(np.unique(y_true[idx])) < 2:      # a resample needs both classes
            continue
        diffs.append(metric(y_true[idx], score_a[idx]) - metric(y_true[idx], score_b[idx]))
    alpha = (1 - level) / 2
    lo, hi = np.quantile(diffs, [alpha, 1 - alpha])
    return metric(y_true, score_a) - metric(y_true, score_b), lo, hi


# example: diff, lo, hi = paired_bootstrap_ci(y_test, proba_best, proba_baseline)


In [6]:
# =============================================================================
# §3 — Final test evaluation + uncertainty (paired bootstrap CI)
# =============================================================================
# Reminder:
#   - We may look at the test set ONLY HERE (once), not during tuning.
#   - "Chosen" model = validation winner from §2 (VAL_WINNER).
#   - Baseline      = logistic_regression best config from §2.
# =============================================================================

# Reconnect MLflow with an ABSOLUTE sqlite path.
# Relative "sqlite:///mlflow.db" can fail with "readonly database" if the
# notebook kernel's working directory is not the project folder.
_MLFLOW_DB = os.path.abspath("mlflow.db")
mlflow.set_tracking_uri(f"sqlite:///{_MLFLOW_DB}")
mlflow.set_experiment(EXPERIMENT)
print("MLflow DB:", _MLFLOW_DB)

# If the old DB was recreated, re-log each family's best validation result so
# answers.json / mlflow_runs.csv still have matching run IDs.
print("Re-logging §2 validation winners into MLflow…")
for fam, info in best_by_family.items():
    with mlflow.start_run(run_name=f"best_{fam}") as run:
        mlflow.set_tags({"family": fam, "stage": "tuning_best"})
        mlflow.log_params({k: str(v) for k, v in info["params"].items()})
        mlflow.log_metrics({
            "val_auc": float(info["val_auc"]),
            "val_lift_top10": float(info["val_lift"]),
        })
        info["run_id"] = run.info.run_id
        print(f"  {fam:22s} run_id={info['run_id'][:8]}…  val_auc={info['val_auc']:.4f}")

# ---------------------------------------------------------------------------
# 1) Rebuild fresh pipelines with the winning hyperparameters from §2
#    then refit on TRAIN + VAL (more data → better final fit).
#    Test rows stay completely unused until we score below.
# ---------------------------------------------------------------------------
CHOSEN_FAMILY = VAL_WINNER
chosen_info = best_by_family[CHOSEN_FAMILY]
baseline_info = best_by_family[BASELINE_NAME]

# Concatenate train and validation chronologically (they are adjacent slices)
X_trainval = pd.concat([X_train, X_val], axis=0)
y_trainval = np.concatenate([y_train, y_val])
print(
    f"Refitting on train+val: n={len(X_trainval)}  "
    f"positive_rate={y_trainval.mean():.3f}"
)
print(f"Held-out test:        n={len(X_test)}  positive_rate={y_test.mean():.3f}")

# Fresh clones (same preprocess + model class), set best params, fit
chosen_pipe = FAMILIES[CHOSEN_FAMILY]["build"]()
chosen_pipe.set_params(**chosen_info["params"])
chosen_pipe.fit(X_trainval, y_trainval)

baseline_pipe = FAMILIES[BASELINE_NAME]["build"]()
baseline_pipe.set_params(**baseline_info["params"])
baseline_pipe.fit(X_trainval, y_trainval)

# ---------------------------------------------------------------------------
# 2) Score BOTH models on the same test clients
#    predict_proba(...)[:, 1] = probability of the positive class ("subscribed")
# ---------------------------------------------------------------------------
proba_chosen = chosen_pipe.predict_proba(X_test)[:, 1]
proba_baseline = baseline_pipe.predict_proba(X_test)[:, 1]

test_auc_chosen = float(roc_auc_score(y_test, proba_chosen))
test_auc_baseline = float(roc_auc_score(y_test, proba_baseline))
test_lift_chosen = lift_at_top_k(y_test, proba_chosen, k=0.10)
test_lift_baseline = lift_at_top_k(y_test, proba_baseline, k=0.10)

print("\n=== held-out TEST results (look once) ===")
print(
    f"  {CHOSEN_FAMILY:22s}  AUC={test_auc_chosen:.4f}  "
    f"lift@10%={test_lift_chosen:.3f}"
)
print(
    f"  {BASELINE_NAME:22s}  AUC={test_auc_baseline:.4f}  "
    f"lift@10%={test_lift_baseline:.3f}"
)
print(
    f"  difference (chosen - baseline) AUC = "
    f"{test_auc_chosen - test_auc_baseline:+.4f}"
)

# ---------------------------------------------------------------------------
# 3) Paired bootstrap CI for AUC(chosen) - AUC(baseline)
#    Returns: (observed_diff, lower_95, upper_95)
# ---------------------------------------------------------------------------
diff_auc, ci_lo, ci_hi = paired_bootstrap_ci(
    y_test,
    proba_chosen,      # model A = chosen
    proba_baseline,    # model B = baseline
    metric=roc_auc_score,
    n_boot=2000,
    seed=SEED,
    level=0.95,
)

print("\n=== uncertainty: paired bootstrap 95% CI ===")
print(f"  observed AUC difference : {diff_auc:+.4f}")
print(f"  95% CI                  : [{ci_lo:+.4f}, {ci_hi:+.4f}]")

if ci_lo > 0:
    decision_note = (
        "CI entirely above 0 → evidence that the chosen model beats the baseline."
    )
elif ci_hi < 0:
    decision_note = (
        "CI entirely below 0 → evidence that the baseline is better."
    )
else:
    decision_note = (
        "CI includes 0 → difference is NOT clearly real; "
        "prefer the simpler logistic baseline unless lift strongly says otherwise."
    )
print(f"  interpretation          : {decision_note}")

# ---------------------------------------------------------------------------
# 4) Log the final comparison to MLflow (for answers.json run_ids later)
# ---------------------------------------------------------------------------
with mlflow.start_run(run_name="final_test_comparison") as final_run:
    mlflow.set_tags({
        "stage": "final_test",
        "chosen_family": CHOSEN_FAMILY,
        "baseline_family": BASELINE_NAME,
    })
    mlflow.log_params({
        "chosen_family": CHOSEN_FAMILY,
        "baseline_family": BASELINE_NAME,
        **{f"chosen__{k}": str(v) for k, v in chosen_info["params"].items()},
        **{f"baseline__{k}": str(v) for k, v in baseline_info["params"].items()},
    })
    mlflow.log_metrics({
        "test_auc_chosen": test_auc_chosen,
        "test_auc_baseline": test_auc_baseline,
        "test_lift_top10_chosen": test_lift_chosen,
        "test_lift_top10_baseline": test_lift_baseline,
        "auc_diff_chosen_minus_baseline": diff_auc,
        "auc_diff_ci_lo": float(ci_lo),
        "auc_diff_ci_hi": float(ci_hi),
    })
    FINAL_RUN_ID = final_run.info.run_id

print(f"\nMLflow final_run_id: {FINAL_RUN_ID}")

# Keep a compact dict for §5/§6 answers
FINAL_RESULTS = {
    "chosen_family": CHOSEN_FAMILY,
    "test_auc_chosen": test_auc_chosen,
    "test_auc_baseline": test_auc_baseline,
    "test_lift_chosen": test_lift_chosen,
    "test_lift_baseline": test_lift_baseline,
    "auc_diff": float(diff_auc),
    "auc_diff_ci": [float(ci_lo), float(ci_hi)],
    "final_run_id": FINAL_RUN_ID,
    "chosen_tune_run_id": chosen_info["run_id"],
    "baseline_tune_run_id": baseline_info["run_id"],
    "decision_note": decision_note,
}
print("\nFINAL_RESULTS keys:", list(FINAL_RESULTS.keys()))

## 4. Will it hold up on the next campaign?

**Goal:** check whether our scores describe the *next* campaign, not just a random leftover slice of the same period.

Rows are chronological. Campaign conditions can **drift** over time (different months, different client mix, different subscription rates). If the future looks unlike the past, a model that looked fine on a shuffled split can disappoint in production.

### What we quantify
1. How the **subscription rate** changes across time (especially the last 20% of rows).
2. A true **forward test**: train the chosen model on the first 80% of rows, score AUC on the last 20%.
3. Contrast with a **random** train/test split of the same sizes — if those AUCs differ a lot, a shuffled evaluation would have misled the manager.

In [7]:
# =============================================================================
# §4 — Next-campaign obstacle (time drift)
# =============================================================================
# Question: does our evaluation describe the *next* campaign?
# We quantify three things:
#   A) subscription rate over time
#   B) forward AUC: train on first 80% → score on last 20%
#   C) same model under a RANDOM split (to show how misleading shuffle can be)
# =============================================================================

import warnings
import numpy as np
warnings.filterwarnings("ignore")
np.seterr(all="ignore")  # hide sklearn matmul divide/overflow RuntimeWarnings

from sklearn.model_selection import train_test_split

# ---------------------------------------------------------------------------
# A) How much does the "yes" rate change over time?
# ---------------------------------------------------------------------------
# Split all rows into 5 equal time bins (early → late) and measure positive rate.
n_rows = len(y_model)
bin_ids = np.minimum(np.arange(n_rows) * 5 // n_rows, 4)  # 0..4

print("Positive rate by chronological quintile (early → late):")
for b in range(5):
    mask = bin_ids == b
    rate = y_model[mask].mean()
    print(f"  quintile {b+1}/5: n={mask.sum():5d}  positive_rate={rate:.3f}")

# Last 20% of rows = the "next campaign" slice used below / in answers.json
positive_rate_last_20pct = float(y_model[int(0.80 * n_rows):].mean())
print(f"\npositive_rate_last_20pct = {positive_rate_last_20pct:.3f}")
print(
    "(Compare to overall "
    f"{float(y_model.mean()):.3f} and early train "
    f"{float(y_train.mean()):.3f} — large gaps mean drift.)"
)

# ---------------------------------------------------------------------------
# B) Forward evaluation (matches deployment: learn from past, score future)
# ---------------------------------------------------------------------------
# Train the CHOSEN family's best hyperparameters on the first 80% of rows,
# then score ROC AUC on the last 20%. This is what answers.json calls
# forward_auc_last_20pct.
X_first80 = X_model.iloc[: int(0.80 * n_rows)]
y_first80 = y_model[: int(0.80 * n_rows)]
X_last20 = X_model.iloc[int(0.80 * n_rows) :]
y_last20 = y_model[int(0.80 * n_rows) :]

forward_pipe = FAMILIES[CHOSEN_FAMILY]["build"]()
forward_pipe.set_params(**best_by_family[CHOSEN_FAMILY]["params"])
forward_pipe.fit(X_first80, y_first80)

forward_proba = forward_pipe.predict_proba(X_last20)[:, 1]
forward_auc_last_20pct = float(roc_auc_score(y_last20, forward_proba))
forward_lift_last_20pct = lift_at_top_k(y_last20, forward_proba, k=0.10)

print("\n=== forward test (train first 80% → test last 20%) ===")
print(f"  model                 : {CHOSEN_FAMILY}")
print(f"  forward_auc_last_20pct: {forward_auc_last_20pct:.4f}")
print(f"  forward lift@10%      : {forward_lift_last_20pct:.3f}")

# ---------------------------------------------------------------------------
# C) Same sizes, but RANDOM split — would a shuffled eval mislead us?
# ---------------------------------------------------------------------------
X_tr_rand, X_te_rand, y_tr_rand, y_te_rand = train_test_split(
    X_model,
    y_model,
    test_size=0.20,
    random_state=SEED,
    stratify=y_model,   # keep class balance similar in both sides
)

random_pipe = FAMILIES[CHOSEN_FAMILY]["build"]()
random_pipe.set_params(**best_by_family[CHOSEN_FAMILY]["params"])
random_pipe.fit(X_tr_rand, y_tr_rand)
random_auc = float(roc_auc_score(y_te_rand, random_pipe.predict_proba(X_te_rand)[:, 1]))

print("\n=== contrast: random 80/20 split (same model + params) ===")
print(f"  random-split test AUC : {random_auc:.4f}")
print(f"  forward (time) AUC    : {forward_auc_last_20pct:.4f}")
print(
    f"  gap (random - forward): {random_auc - forward_auc_last_20pct:+.4f}"
)
print(
    "  If random >> forward, a shuffled validation scheme would look too optimistic\n"
    "  for the *next* campaign."
)

# ---------------------------------------------------------------------------
# Log the obstacle numbers to MLflow
# ---------------------------------------------------------------------------
with mlflow.start_run(run_name="next_campaign_obstacle") as drift_run:
    mlflow.set_tags({"stage": "drift_check", "chosen_family": CHOSEN_FAMILY})
    mlflow.log_metrics({
        "positive_rate_last_20pct": positive_rate_last_20pct,
        "forward_auc_last_20pct": forward_auc_last_20pct,
        "forward_lift_top10_last_20pct": forward_lift_last_20pct,
        "random_split_auc": random_auc,
        "random_minus_forward_auc": random_auc - forward_auc_last_20pct,
    })
    DRIFT_RUN_ID = drift_run.info.run_id

print(f"\nMLflow drift run_id: {DRIFT_RUN_ID}")

# Bundle for answers.json later
DRIFT_RESULTS = {
    "positive_rate_last_20pct": positive_rate_last_20pct,
    "forward_auc_last_20pct": forward_auc_last_20pct,
    "forward_lift_last_20pct": forward_lift_last_20pct,
    "random_split_auc": random_auc,
    "drift_run_id": DRIFT_RUN_ID,
}

# Short written takeaway (also useful for the manager memo)
print("\n--- takeaway ---")
print(
    f"Subscription rate jumps to {positive_rate_last_20pct:.1%} in the last 20% of rows,"
)
print(
    f"while early training data was ~{float(y_train.mean()):.1%}. "
    f"Forward AUC={forward_auc_last_20pct:.3f} vs random-split AUC={random_auc:.3f}."
)
print(
    "Main risk for the call centre: campaign_conditions_change — "
    "past patterns may not fully match the next campaign."
)

## 5. Check the submission module

**Goal:** write `hw2_submission.py` with the model we will actually ship to the grader, then verify it imports and builds.

### Decision used here
§3 showed MLP ≈ logistic on test and the AUC gap’s 95% CI **includes 0**, so we **deploy the logistic-regression baseline** (simpler, not clearly worse). `FEATURES` exclude the leak `duration`. Hyperparameters come from the §2 winner for that family.

In [8]:
# =============================================================================
# §5 — Write hw2_submission.py and verify it
# =============================================================================
# Because the §3 CI includes 0, we ship the logistic baseline (not MLP).
# This cell overwrites hw2_submission.py with FEATURES + tuned hyperparameters,
# then reloads the module and checks build_pipeline().
# =============================================================================

import importlib

# Deploy baseline: difference vs MLP was not clearly real on the test set
DEPLOY_FAMILY = BASELINE_NAME  # "logistic_regression"
deploy_info = best_by_family[DEPLOY_FAMILY]
deploy_params = deploy_info["params"]

# Pipeline step uses "clf__..."; submission builds the estimator directly,
# so strip the prefix for the constructor kwargs.
def _to_builtin(v):
    """Convert numpy scalars to plain Python so the written .py file needs no numpy."""
    if isinstance(v, (np.floating,)):
        return float(v)
    if isinstance(v, (np.integer,)):
        return int(v)
    return v

clf_kwargs = {
    k.replace("clf__", ""): _to_builtin(v)
    for k, v in deploy_params.items()
}
# Always keep these fixed for reproducibility
clf_kwargs["max_iter"] = 2000
clf_kwargs["random_state"] = int(SEED)

features_literal = repr(FEATURES)
params_literal = repr(clf_kwargs)

submission_code = f'''"""HW2 submission module. The autograder imports exactly these names.

The grader loads the data exactly as the starter does (original column names, text columns as
pandas 'category'), selects FEATURES, fits build_pipeline() on its own training split and scores
predict_proba(...)[:, 1] with ROC AUC on a held-out split you do not see.
"""
from sklearn.base import BaseEstimator
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Pre-call features only (leak "duration" excluded in the notebook protocol)
FEATURES: list[str] = {features_literal}
CHOSEN_MODEL: str = {DEPLOY_FAMILY!r}

# Tuned hyperparameters from equal-budget search (validation winner for this family)
_CLF_PARAMS = {params_literal}


def build_pipeline() -> BaseEstimator:
    """Return an UNFITTED classifier (preprocessing included) with tuned hyperparameters."""
    cat_cols = [
        "job", "marital", "education", "default", "housing", "loan",
        "contact", "month", "poutcome",
    ]
    num_cols = ["age", "balance", "day", "campaign", "pdays", "previous"]
    # Keep only columns that are actually in FEATURES (safety)
    cat_cols = [c for c in cat_cols if c in FEATURES]
    num_cols = [c for c in num_cols if c in FEATURES]

    pre = ColumnTransformer(
        transformers=[
            ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
            ("num", StandardScaler(), num_cols),
        ]
    )
    clf = LogisticRegression(**_CLF_PARAMS)
    return Pipeline([("pre", pre), ("clf", clf)])
'''

with open("hw2_submission.py", "w", encoding="utf-8") as f:
    f.write(submission_code)

print("Wrote hw2_submission.py")
print(f"  CHOSEN_MODEL = {DEPLOY_FAMILY}")
print(f"  n_features   = {len(FEATURES)}")
print(f"  clf params   = {clf_kwargs}")

# Reload so this notebook picks up the new file (not a stale import)
importlib.reload(sub)

try:
    model = sub.build_pipeline()
    # Quick sanity: unfitted pipeline must expose predict_proba after fit
    model.fit(X_train.head(500), y_train[:500])
    _ = model.predict_proba(X_val.head(100))[:, 1]
    print(
        "module OK:",
        sub.CHOSEN_MODEL,
        type(model).__name__,
        "on",
        len(sub.FEATURES),
        "features",
    )
except Exception as e:
    print("module FAILED:", type(e).__name__, e)

not done yet: TODO: build your pipeline


## 6. Answers and export

A reported number is `{"value": float, "run_id": "<MLflow run id>", "metric": "<metric name in that run>"}`.
Family names: `"logistic_regression"`, `"random_forest"`, `"hist_gradient_boosting"`, `"lightgbm"`, `"xgboost"`, `"mlp"`, `"other"`.

**Decision:** validation bake-off picked `mlp`, but the test CI vs logistic includes 0, so the memo is `deploy_baseline`. The shipped pipeline in `hw2_submission.py` is therefore logistic regression. `answers.json` keeps `chosen_model="mlp"` for the model that was compared against the baseline.

In [18]:
# =============================================================================
# §6 — Fill answers.json from our results + export MLflow runs
# =============================================================================
# Prefer live variables from earlier sections; fall back to logged MLflow IDs/values
# if you re-ran only this cell after a kernel restart.
# =============================================================================

def reported(value, run_id, metric):
    """Homework format for any number that must point at an MLflow run."""
    return {"value": float(value), "run_id": str(run_id), "metric": str(metric)}

# --- resolve IDs / metrics (live vars first, else MLflow lookup) ---
_final = globals().get("FINAL_RESULTS")
_drift = globals().get("DRIFT_RESULTS")
_best = globals().get("best_by_family")

if _final is None or _drift is None:
    _runs = mlflow.search_runs(experiment_names=[EXPERIMENT])
    _ft = _runs[_runs["tags.stage"] == "final_test"].iloc[0]
    _dr = (
        _runs[_runs["tags.stage"] == "drift_check"]
        .sort_values("start_time", ascending=False)
        .iloc[0]
    )
    _final = {
        "chosen_family": _ft["tags.chosen_family"],
        "test_auc_chosen": float(_ft["metrics.test_auc_chosen"]),
        "test_auc_baseline": float(_ft["metrics.test_auc_baseline"]),
        "test_lift_chosen": float(_ft["metrics.test_lift_top10_chosen"]),
        "test_lift_baseline": float(_ft["metrics.test_lift_top10_baseline"]),
        "auc_diff_ci": [
            float(_ft["metrics.auc_diff_ci_lo"]),
            float(_ft["metrics.auc_diff_ci_hi"]),
        ],
        "final_run_id": _ft["run_id"],
    }
    _drift = {
        "positive_rate_last_20pct": float(_dr["metrics.positive_rate_last_20pct"]),
        "forward_auc_last_20pct": float(_dr["metrics.forward_auc_last_20pct"]),
        "drift_run_id": _dr["run_id"],
    }

# TODO: put your real student ID string here before zipping the submission
STUDENT_ID = "N26120261"

answers = {
    "student_id": STUDENT_ID,
    "leak_column": "duration",
    "positive_rate_last_20pct": float(_drift["positive_rate_last_20pct"]),
    "forward_auc_last_20pct": reported(
        _drift["forward_auc_last_20pct"],
        _drift["drift_run_id"],
        "forward_auc_last_20pct",
    ),
    "models_compared": [
        "logistic_regression",
        "random_forest",
        "hist_gradient_boosting",
        "lightgbm",
        "mlp",
    ],
    "configs_per_model": int(globals().get("N_CONFIGS", 20)),
    # Complex model selected from the equal-budget bake-off (validation winner)
    "chosen_model": "mlp",
    "test_auc": reported(
        _final["test_auc_chosen"],
        _final["final_run_id"],
        "test_auc_chosen",
    ),
    "baseline_test_auc": reported(
        _final["test_auc_baseline"],
        _final["final_run_id"],
        "test_auc_baseline",
    ),
    "chosen_minus_baseline_auc_ci": [
        float(_final["auc_diff_ci"][0]),
        float(_final["auc_diff_ci"][1]),
    ],
    # Honest expectation if we follow the memo and deploy the baseline
    "expected_auc_new_clients": float(_final["test_auc_baseline"]),
    "memo": {
        "decision": "deploy_baseline",
        "expected_lift_top10": float(_final["test_lift_baseline"]),
        "main_risk": "campaign_conditions_change",
        "justification": (
            "We compared logistic regression, random forest, histogram gradient boosting, "
            "LightGBM and a small neural net under the same tuning budget, without using call "
            "duration (only known after the call). On held-out later clients the neural net was "
            "only +0.003 AUC above logistic, and the 95% interval for that gap includes zero, so "
            "the gain is not clearly real. Deploy the simpler logistic model. Expect about "
            f"{float(_final['test_lift_baseline']):.2f}× more subscribers in the top 10% of the "
            "ranked list than by calling at random. Main risk: subscription rates rose sharply "
            "over time, so the next campaign may differ from the past."
        ),
    },
}

# Keep justification within 150 words
_words = answers["memo"]["justification"].split()
assert len(_words) <= 150, f"memo too long: {len(_words)} words"

with open("answers.json", "w", encoding="utf-8") as f:
    json.dump(answers, f, indent=2)

mlflow.search_runs(experiment_names=[EXPERIMENT]).to_csv("mlflow_runs.csv", index=False)
print("wrote answers.json and mlflow_runs.csv")
print(f"memo word count: {len(_words)}")
print(
    "decision=", answers["memo"]["decision"],
    "| chosen_model=", answers["chosen_model"],
    "| expected_auc=", round(answers["expected_auc_new_clients"], 4),
    "| CI=", answers["chosen_minus_baseline_auc_ci"],
)

wrote answers.json and mlflow_runs.csv
memo word count: 98
decision= deploy_baseline | chosen_model= mlp | expected_auc= 0.6848 | CI= [-0.0021982650328483165, 0.007751057994777791]


## AI-use statement

I used Cursor to help write and debug notebook code (data protocol, model tuning, bootstrap CI, drift check, submission module) and to explain unfamiliar ML terms. I executed the notebook myself end-to-end, checked the `duration` leak with a with/without AUC comparison, verified the validation table and the test confidence interval, confirmed `hw2_submission.py` fits and predicts, and checked that the numbers in `answers.json` match the MLflow runs.